# Subject: cmdecorators
Source: /home/devuser/edu_data/datasets/TrainingDB/EncryT/secretpy/secretpy/cmdecorators

### File: block.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

from .decorator import AbstractMachineDecorator


class Block(AbstractMachineDecorator):

    def __init__(self, machine, length=5, sep=" "):
        self.length = length
        self.sep = sep
        super(Block, self).__init__(machine)

    def encrypt(self, text):
        txt = self._machine.encrypt(text.lower())
        return self.sep.join(txt[i:i + self.length] for i in range(0, len(txt), self.length))

    def decrypt(self, text):
        # remove separator
        step = self.length + len(self.sep)
        txt = "".join(text[i:i + self.length] for i in range(0, len(text), step))
        return self._machine.decrypt(txt.lower())

### File: decorator.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

from ..abstractmachine import AbstractCryptMachine
from secretpy import alphabets


class AbstractMachineDecorator(AbstractCryptMachine):
    def __init__(self, machine):
        self._machine = machine

    def set_key(self, key):
        self._machine.set_key(key)

    def set_alphabet(self, alphabet=alphabets.ENGLISH):
        self._machine.set_alphabet(alphabet)

    def get_alphabet(self):
        return self._machine.get_alphabet()

    def get_crypt_alphabet(self):
        return self._machine.get_crypt_alphabet()

    def set_cipher(self, cipher):
        self._machine.set_cipher(cipher)

    def has_mixedcase(self):
        return self._machine.has_mixedcase()

    def encrypt(self, text):
        pass

    def decrypt(self, text):
        pass

### File: numbers.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
from .decorator import AbstractMachineDecorator


class Numbers(AbstractMachineDecorator):

    def __init__(self, machine, sep=' '):
        self.__sep = sep
        super(Numbers, self).__init__(machine)

    def encrypt(self, text):
        res = self._machine.encrypt(text)
        alphabet = self._machine.get_crypt_alphabet()
        subst = {t: str(i) for i, letters in enumerate(alphabet) for t in letters}
        return self.__sep.join(subst[t] for t in res)

    def decrypt(self, text):
        alphabet = self._machine.get_alphabet()
        subst = {str(i): letters[0] for i, letters in enumerate(alphabet)}
        # convert numbers to the characters of the alphabet
        txt = "".join(subst[i] for i in text.split(self.__sep))
        return self._machine.decrypt(txt)

### File: save_all.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

from .decorator import AbstractMachineDecorator


class SaveAll(AbstractMachineDecorator):

    def encrypt(self, text):
        alphabet = self._machine.get_alphabet()
        return self.__crypt(text, self._machine.encrypt, alphabet)

    def decrypt(self, text):
        alphabet = self._machine.get_crypt_alphabet()
        return self.__crypt(text, self._machine.decrypt, alphabet)

    def __crypt(self, text, func, alphabet):
        # make lower case and save indexes
        txt = text
        if not self._machine.has_mixedcase():
            upcases = [i for i, c in enumerate(text) if c.isupper()]
            txt = txt.lower()

        # prepare alphabet
        alpha = {c: 1 for letters in alphabet for c in letters}

        # save indexes of non-alphabet characters
        chars = [i for i, c in enumerate(txt) if c not in alpha]

        # execute function
        res = list(func(txt))

        # restore non-alphabet characters
        for i in chars:
            res.insert(i, text[i])

        # restore uppercase
        if not self._machine.has_mixedcase():
            for i in filter(lambda x: x < len(res), upcases):
                res[i] = res[i].upper()

        return "".join(res)

### File: savecase.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

from .decorator import AbstractMachineDecorator


class SaveCase(AbstractMachineDecorator):

    def encrypt(self, text):
        return self.__crypt(text, self._machine.encrypt)

    def decrypt(self, text):
        return self.__crypt(text, self._machine.decrypt)

    def __crypt(self, text, func):
        uppercases = [i for i, char in enumerate(text) if char == char.upper()]
        text = text.lower()

        res = func(text)
        res = list(res)

        for i in uppercases:
            res[i] = res[i].upper()
        return u"".join(res)

### File: uppercase.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

from .decorator import AbstractMachineDecorator


class UpperCase(AbstractMachineDecorator):

    def encrypt(self, text):
        return self._machine.encrypt(text).upper()

    def decrypt(self, text):
        return self._machine.decrypt(text).upper()